In [ ]:
import numpy as np
from PIL import Image

def gcd(p, q):
    if q == 0:
        return p, 1, 0
    d, a, b = gcd(q, p % q)
    return d, b, a - (p // q) * b

def inverse(k, n):
    d, a, b = gcd(k, n)
    if d > 1:
        print("Inverse does not exist.")
        return 0
    if a > 0:
        return a
    return n + a

def main():
    # Open the input PNG image
    input_image_path = "modi.jpg"
    img = Image.open(input_image_path)
    img_array = np.array(img)

    # Flatten the image array
    a1 = img_array.flatten()

    len1 = len(a1)
    print(len1)

    B1 = np.zeros(len1, dtype=np.uint8)
    B2 = np.zeros(len1, dtype=np.uint8)
    B3 = np.zeros(len1, dtype=np.uint8)

    for i in range(len1):
        if i < 500:
            B1[i] = a1[i]
            B2[i] = a1[i]
            B3[i] = a1[i]
        else:
            B1[i] = a1[i] % 3
            B2[i] = a1[i] % 5
            B3[i] = a1[i] % 17

    # Convert the arrays back to images and save as PNG files
    out_image_paths = ["out.png", "out1.png", "out2.png"]
    for i, data in enumerate([B1, B2, B3]):
        # Reshape the data to the original image shape
        data = data.reshape(img_array.shape)

        # Create PIL image from array
        new_img = Image.fromarray(data)

        # Save the image
        new_img.save(out_image_paths[i])

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
from PIL import Image

def gcd(p, q):
    if q == 0:
        return p, 1, 0
    d, a, b = gcd(q, p % q)
    return d, b, a - (p // q) * b

def inverse(k, n):
    d, a, _ = gcd(k, n)
    if d > 1:
        print("Inverse does not exist.")
        return 0
    if a < 0:
        return n + a
    return a

def main():
    try:
        with open("out.png", "rb") as in1, open("out1.png", "rb") as in2, open("out2.png", "rb") as in3:
            img1 = Image.open(in1)
            img2 = Image.open(in2)
            img3 = Image.open(in3)

            # Convert images to numpy arrays
            A1 = np.array(img1)
            A2 = np.array(img2)
            A3 = np.array(img3)

            len1 = len(A1)
            len2 = len(A2)
            len3 = len(A3)
            print(len1, len2, len3)

            m1 = 255 // 3
            m2 = 255 // 5
            m3 = 255 // 17
            w1 = inverse(m1, 3)
            w2 = inverse(m2, 5)
            w3 = inverse(m3, 17)

            # Perform decryption
            Y = []
            for i in range(len1):
                if i < 500:
                    Y.append(A1[i])
                else:
                    Y.append((A1[i] * w1 + A2[i] * w2 + A3[i] * w3) % 255)

            # Create a new PNG image from the decrypted data
            dec_image = Image.fromarray(np.array(Y, dtype=np.uint8))

            # Save the decrypted image
            dec_image.save("dec.png")
    except Exception as er:
        print(er)

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
from PIL import Image

# Function to encrypt an image using RNS
def encrypt_image(image_path, moduli):
    # Load image
    img = Image.open(image_path)
    img_array = np.array(img)
    
    # Encrypt each channel separately
    encrypted_img_arrays = []
    for channel in range(3):
        encrypted_img_array = np.zeros_like(img_array[..., channel], dtype=np.int32)
        for k, modulus in enumerate(moduli):
            encrypted_img_array += (img_array[..., channel] % modulus) * np.prod(moduli) // modulus
        encrypted_img_array %= np.prod(moduli)
        encrypted_img_arrays.append(encrypted_img_array.astype(np.uint8))
    
    return encrypted_img_arrays

# Function to decrypt an image using RNS
def decrypt_image(encrypted_img_arrays, moduli):
    # Calculate product of moduli
    prod_moduli = np.prod(moduli)
    
    # Decrypt each channel separately
    decrypted_img_array = np.zeros(encrypted_img_arrays[0].shape[:2] + (3,), dtype=np.uint8)
    for i in range(3):
        decrypted_img_array[..., i] = np.sum(encrypted_img_arrays[i] * np.prod(moduli) // moduli[i], axis=-1) % prod_moduli
    
    return decrypted_img_array


# Define moduli for the RNS system
moduli_encrypt = [251, 257, 263]  # Example moduli for encryption
moduli_decrypt = [3, 5, 17]  # Example moduli for decryption

# Encryption and decryption process
original_image_path = input("Enter the path to the image file: ")

# Encrypt original image
encrypted_img_arrays = encrypt_image(original_image_path, moduli_encrypt)
encrypted_images = [Image.fromarray(arr) for arr in encrypted_img_arrays]

# Decrypt encrypted image
decrypted_img_array = decrypt_image(encrypted_img_arrays, moduli_decrypt)
decrypted_img = Image.fromarray(decrypted_img_array)

# Display encrypted and decrypted images
for i, img in enumerate(encrypted_images):
    img.show(title=f"Encrypted Channel {i+1}")
decrypted_img.show(title="Decrypted Image")



In [ ]:
import numpy as np
from PIL import Image

# Function to encrypt an image channel using RNS
def encrypt_channel(channel_data, moduli):
    encrypted_channel_data = np.zeros(channel_data.shape + (len(moduli),), dtype=np.uint8)
    for k, modulus in enumerate(moduli):
        encrypted_channel_data[..., k] = channel_data % modulus
    return encrypted_channel_data

# Function to decrypt an image channel using RNS
def decrypt_channel(encrypted_channel_data, moduli):
    # Calculate product of moduli
    prod_moduli = int(np.prod(moduli))  # Convert to native Python int
    
    # Decrypt each pixel value
    decrypted_channel_data = np.zeros(encrypted_channel_data.shape[:-1], dtype=np.uint64)
    for k, modulus in enumerate(moduli):
        modulus = int(modulus)  # Convert to native Python int
        p = prod_moduli // modulus
        inv_p = pow(p, -1, modulus)  # Ensure p is a native Python int
        decrypted_channel_data += encrypted_channel_data[..., k] * p * inv_p
    decrypted_channel_data %= prod_moduli
    
    return decrypted_channel_data.astype(np.uint8)


# Function to encrypt an image using RNS for each RGB channel
def encrypt_image_rns(image_path, moduli_r, moduli_g, moduli_b):
    img = Image.open(image_path)
    img_array = np.array(img)
    
    # Encrypt R, G, B channels separately
    r_encrypted = encrypt_channel(img_array[:, :, 0], moduli_r)
    g_encrypted = encrypt_channel(img_array[:, :, 1], moduli_g)
    b_encrypted = encrypt_channel(img_array[:, :, 2], moduli_b)
    
    return r_encrypted, g_encrypted, b_encrypted

# Function to decrypt the image from the encrypted RGB channels
def decrypt_image_rns(r_encrypted, g_encrypted, b_encrypted, moduli_r, moduli_g, moduli_b):
    r_decrypted = decrypt_channel(r_encrypted, moduli_r)
    g_decrypted = decrypt_channel(g_encrypted, moduli_g)
    b_decrypted = decrypt_channel(b_encrypted, moduli_b)
    
    # Combine R, G, B channels
    decrypted_img_array = np.stack((r_decrypted, g_decrypted, b_decrypted), axis=-1)
    
    return decrypted_img_array

# Define moduli for each RGB channel
moduli_r = [251, 257, 263]
moduli_g = [269, 271, 277]
moduli_b = [281, 283, 293]

# Encryption process
original_image_path = "modi.jpg"
r_encrypted, g_encrypted, b_encrypted = encrypt_image_rns(original_image_path, moduli_r, moduli_g, moduli_b)

# Save encrypted channels as separate images
Image.fromarray(r_encrypted[..., 0], 'L').save('r_encrypted_image.png')
Image.fromarray(g_encrypted[..., 0], 'L').save('g_encrypted_image.png')
Image.fromarray(b_encrypted[..., 0], 'L').save('b_encrypted_image.png')

# Decryption process
decrypted_img_array = decrypt_image_rns(r_encrypted, g_encrypted, b_encrypted, moduli_r, moduli_g, moduli_b)
decrypted_img = Image.fromarray(decrypted_img_array)
decrypted_img.save('decrypted_image.png')



In [ ]:
#############encryption##################

In [ ]:
from PIL import Image
import numpy as np

def encrypt_image(image_path):
    # Load the image
    img = Image.open(image_path)
    img_array = np.array(img)

    # Initialize arrays to store encrypted images
    encrypted_images = [np.zeros_like(img_array) for _ in range(3)]

    # Process each pixel
    for i in range(img_array.shape[0]):
        for j in range(img_array.shape[1]):
            # If it's within the first 500 pixels, copy pixel values directly
            if i * img_array.shape[1] + j < 500:
                for k in range(3):
                    encrypted_images[k][i, j] = img_array[i, j]
            else:
                # Otherwise, compute residues modulo 3, 5, and 17 for each pixel
                encrypted_images[0][i, j] = img_array[i, j] % 3
                encrypted_images[1][i, j] = img_array[i, j] % 5
                encrypted_images[2][i, j] = img_array[i, j] % 17

    # Write the encrypted images to separate files
    for idx, encrypted_img in enumerate(encrypted_images):
        Image.fromarray(encrypted_img).save(f'encrypted_image_mod_{idx}.png')

# Replace 'path_to_image.jpg' with the path to your image file
encrypt_image('Images_for_testing/bollyw.jpg')


In [ ]:
from PIL import Image
import numpy as np

# Function to calculate the multiplicative inverse
def multiplicative_inverse(a, m):
    m0, x0, x1 = m, 0, 1
    if m == 1:
        return 0
    while a > 1:
        q = a // m
        m, a = a % m, m
        x0, x1 = x1 - q * x0, x0
    return x1 + m0 if x1 < 0 else x1

# Function to apply the Chinese Remainder Theorem for decryption
def crt(pixels, moduli):
    sum = 0
    prod = np.prod(moduli)
    for pixel, modulus in zip(pixels, moduli):
        p = prod // modulus
        sum += pixel * multiplicative_inverse(p, modulus) * p
    return sum % prod

# Function to decrypt the image
def decrypt_image(encrypted_paths, moduli):
    # Read the pixel values from each encrypted image
    encrypted_images = [np.array(Image.open(path)) for path in encrypted_paths]

    # Initialize array to store the decrypted image
    decrypted_img_array = np.zeros_like(encrypted_images[0])

    # Decrypt each pixel value using the CRT formula
    for i in range(decrypted_img_array.shape[0]):
        for j in range(decrypted_img_array.shape[1]):
            pixels = [img[i, j] for img in encrypted_images]
            decrypted_img_array[i, j] = crt(pixels, moduli)

    # Write the decrypted pixel values to an output file
    decrypted_img = Image.fromarray(decrypted_img_array.astype(np.uint8))
    decrypted_img.save('decrypted_image.png')

# Define the paths to your encrypted images and the moduli used for encryption
encrypted_paths = ['encrypted_image_mod_0.png', 'encrypted_image_mod_1.png', 'encrypted_image_mod_2.png']
moduli = [3, 5, 17]

# Call the function to decrypt the image
decrypt_image(encrypted_paths, moduli)


In [ ]:
#####################decrypt#################

from PIL import Image
import numpy as np

# Function to calculate the multiplicative inverse using the Extended Euclidean Algorithm
def multiplicative_inverse(a, m):
    m0, x0, x1 = m, 0, 1
    if m == 1:
        return 0
    while a > 1:
        q = a // m
        m, a = a % m, m
        x0, x1 = x1 - q * x0, x0
    return x1 + m0 if x1 < 0 else x1

# Function to apply the Chinese Remainder Theorem for decryption
def crt(pixels, moduli):
    total = 0
    prod = np.prod(moduli, dtype=np.uint64)
    for pixel, modulus in zip(pixels, moduli):
        p = prod // modulus
        inv = multiplicative_inverse(p, modulus)
        total += pixel * inv * p
    return total % prod

# Function to decrypt the image
def decrypt_image(encrypted_paths, moduli):
    # Read the pixel values from each encrypted image
    encrypted_images = [np.array(Image.open(path), dtype=np.uint64) for path in encrypted_paths]

    # Initialize array to store the decrypted image
    decrypted_img_array = np.zeros_like(encrypted_images[0])

    # Decrypt each pixel value using the CRT formula
    for i in range(decrypted_img_array.shape[0]):
        for j in range(decrypted_img_array.shape[1]):
            pixels = [img[i, j] for img in encrypted_images]
            decrypted_img_array[i, j] = crt(pixels, moduli)

    # Convert decrypted pixel values to uint8
    decrypted_img_array = decrypted_img_array.astype(np.uint8)
    
    # Write the decrypted pixel values to an output file
    decrypted_img = Image.fromarray(decrypted_img_array)
    decrypted_img.save('decrypted_image.png')

# Define the paths to your encrypted images and the moduli used for encryption
encrypted_paths = ['encrypted_image_mod_0.png', 'encrypted_image_mod_1.png', 'encrypted_image_mod_2.png']
moduli = [3, 5, 17]

# Call the function to decrypt the image
decrypt_image(encrypted_paths, moduli)


In [ ]:
##################encryption##############


from PIL import Image
import numpy as np

def encrypt_image(image_path):
    # Load the image
    img = Image.open(image_path)
    img_array = np.array(img)

    # Initialize arrays to store encrypted images
    encrypted_images = [np.zeros_like(img_array) for _ in range(3)]

    # Process each pixel
    for i in range(img_array.shape[0]):
        for j in range(img_array.shape[1]):
            # If it's within the first 500 pixels, copy pixel values directly
            if i * img_array.shape[1] + j < 500:
                for k in range(3):
                    encrypted_images[k][i, j] = img_array[i, j]
            else:
                # Otherwise, compute residues modulo 3, 5, and 17 for each pixel
                encrypted_images[0][i, j] = img_array[i, j] % 3
                encrypted_images[1][i, j] = img_array[i, j] % 5
                encrypted_images[2][i, j] = img_array[i, j] % 17

    # Write the encrypted images to separate files
    for idx, encrypted_img in enumerate(encrypted_images):
        Image.fromarray(encrypted_img).save(f'encrypted_image_mod_{idx}.png')

# Replace 'path_to_image.jpg' with the path to your image file
encrypt_image('Images_for_testing/bollyw.jpg')

In [ ]:
from PIL import Image
import numpy as np

def encrypt_image(image_path):
    # Load the image
    img = Image.open(image_path)
    img_array = np.array(img)

    # Initialize arrays to store encrypted images
    encrypted_images = [np.zeros_like(img_array) for _ in range(3)]

    # Process each pixel
    for i in range(img_array.shape[0]):
        for j in range(img_array.shape[1]):
            # If it's within the first 500 pixels, copy pixel values directly
            if i * img_array.shape[1] + j < 500:
                for k in range(3):
                    encrypted_images[k][i, j] = img_array[i, j]
            else:
                # Otherwise, compute residues modulo 3, 5, and 17 for each pixel
                encrypted_images[0][i, j] = img_array[i, j] % 3
                encrypted_images[1][i, j] = img_array[i, j] % 5
                encrypted_images[2][i, j] = img_array[i, j] % 17

    # Write the encrypted images to separate files
    for idx, encrypted_img in enumerate(encrypted_images):
        Image.fromarray(encrypted_img).save(f'encrypted_{image_path}_mod_{idx}.png')

# Define the dictionary containing images with detected persons
# images_with_persons = {
#     'akshaykumar.jpg': ['akshay kumar.jpg'],
#     'bollyw.jpg': ['ranveer singh.jpg', 'akshay kumar.jpg'],
#     'modi1.jpg': ['ranveer singh.jpg', 'modi.jpg']
# }

# Encrypt images where persons were detected
for image_path, detected_persons in images_with_persons.items():
    print(f"Encrypting {image_path}...")
    encrypt_image(f"Images_for_testing/{image_path}")


In [ ]:
##############decrypt

from PIL import Image
import numpy as np

# Function to calculate the multiplicative inverse using the Extended Euclidean Algorithm
def multiplicative_inverse(a, m):
    m0, x0, x1 = m, 0, 1
    if m == 1:
        return 0
    while a > 1:
        q = a // m
        m, a = a % m, m
        x0, x1 = x1 - q * x0, x0
    return x1 + m0 if x1 < 0 else x1

# Function to apply the Chinese Remainder Theorem for decryption
def crt(pixels, moduli):
    total = 0
    prod = np.prod(moduli, dtype=np.uint64)
    for pixel, modulus in zip(pixels, moduli):
        p = prod // modulus
        inv = multiplicative_inverse(p, modulus)
        total += pixel * inv * p
    return total % prod

# Function to decrypt the image
def decrypt_image(encrypted_paths, moduli):
    # Read the pixel values from each encrypted image
    encrypted_images = [np.array(Image.open(path), dtype=np.uint64) for path in encrypted_paths]

    # Initialize array to store the decrypted image
    decrypted_img_array = np.zeros_like(encrypted_images[0])

    # Decrypt each pixel value using the CRT formula
    for i in range(decrypted_img_array.shape[0]):
        for j in range(decrypted_img_array.shape[1]):
            pixels = [img[i, j] for img in encrypted_images]
            decrypted_img_array[i, j] = crt(pixels, moduli)

    # Convert decrypted pixel values to uint8
    decrypted_img_array = decrypted_img_array.astype(np.uint8)
    
    # Write the decrypted pixel values to an output file
    decrypted_img = Image.fromarray(decrypted_img_array)
    decrypted_img.save('decrypted_image.png')

# Define the paths to your encrypted images and the moduli used for encryption
encrypted_paths = ['encrypted_image_mod_0.png', 'encrypted_image_mod_1.png', 'encrypted_image_mod_2.png']
moduli = [3, 5, 17]

# Call the function to decrypt the image
decrypt_image(encrypted_paths, moduli)


In [ ]:
import os
from PIL import Image
import numpy as np

# Function to calculate the multiplicative inverse using the Extended Euclidean Algorithm
def multiplicative_inverse(a, m):
    m0, x0, x1 = m, 0, 1
    if m == 1:
        return 0
    while a > 1:
        q = a // m
        m, a = a % m, m
        x0, x1 = x1 - q * x0, x0
    return x1 + m0 if x1 < 0 else x1

# Function to apply the Chinese Remainder Theorem for decryption
def crt(pixels, moduli):
    total = 0
    prod = np.prod(moduli, dtype=np.uint64)
    for pixel, modulus in zip(pixels, moduli):
        p = prod // modulus
        inv = multiplicative_inverse(p, modulus)
        total += pixel * inv * p
    return total % prod

# Function to decrypt a single image
def decrypt_image(encrypted_paths, moduli):
    # Read the pixel values from each encrypted image
    encrypted_images = [np.array(Image.open(path), dtype=np.uint64) for path in encrypted_paths]

    # Initialize array to store the decrypted image
    decrypted_img_array = np.zeros_like(encrypted_images[0])

    # Decrypt each pixel value using the CRT formula
    for i in range(decrypted_img_array.shape[0]):
        for j in range(decrypted_img_array.shape[1]):
            pixels = [img[i, j] for img in encrypted_images]
            decrypted_img_array[i, j] = crt(pixels, moduli)

    # Convert decrypted pixel values to uint8
    decrypted_img_array = decrypted_img_array.astype(np.uint8)
    
    # Return the decrypted image as a PIL image
    return Image.fromarray(decrypted_img_array)

# Function to decrypt images in a specified folder based on the provided filename
def decrypt_folder_for_filename(folder_path, filename):
    folder_path = os.path.join('encrypted_images', filename)
    encrypted_paths = [os.path.join(folder_path, f'encrypted_mod_{i}.png') for i in range(3)]
    moduli = [3, 5, 17]
    decrypted_img = decrypt_image(encrypted_paths, moduli)
    decrypted_img.save(os.path.join(folder_path, 'decrypted_image.png'))

# Specify the filename for which you want to decrypt images
filename = 'bollyw'

# Call the function to decrypt images in the folder corresponding to the provided filename
decrypt_folder_for_filename('encrypted_images', filename)
